# 06. Глубокий анализ корреляций, причинности и управляемости

Цели:
1. Полная корреляционная матрица между всеми тегами АВТ и 24-2000
2. Анализ временных лагов и причинно-следственных связей (Granger causality)
3. Идентификация режимов работы и их устойчивости
4. Оценка управляемости: какие параметры контролируются, какие — следствия
5. Анализ чувствительности: как изменения входов влияют на выходы
6. Динамическая стабильность и дрейф параметров

**Важно**: Анализ проводится на данных without LIMS/PAK, поэтому качественные показатели оценить нельзя.

In [ ]:
from pathlib import Path
import sys
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from IPython.display import display, Markdown

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

HERE = Path.cwd().resolve()
EDA_DIR = HERE if HERE.name == 'eda' else HERE / 'eda'
DATA_DIR = EDA_DIR.parent / 'data'
ARTIFACTS = EDA_DIR / 'artifacts'
ARTIFACTS.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(EDA_DIR))

from eda_utils import load_telemetry

print('DATA_DIR:', DATA_DIR)

## 1. Загрузка и подготовка данных

In [ ]:
# Загрузка данных
avt = load_telemetry(DATA_DIR / 'avt_tags.csv')
hydro = load_telemetry(DATA_DIR / '242000_tags.csv')

print(f'AVT shape: {avt.shape}')
print(f'242000 shape: {hydro.shape}')
print(f'\nAVT period: {avt["date"].min()} to {avt["date"].max()}')
print(f'242000 period: {hydro["date"].min()} to {hydro["date"].max()}')

# Объединение данных по времени
avt_clean = avt.set_index('date').sort_index()
hydro_clean = hydro.set_index('date').sort_index()

# Префиксы для различения источников
avt_clean.columns = ['AVT_' + col for col in avt_clean.columns]
hydro_clean.columns = ['H24_' + col for col in hydro_clean.columns]

# Объединение с допуском ±5 минут
combined = pd.merge_asof(
    avt_clean.reset_index(), 
    hydro_clean.reset_index(),
    on='date',
    tolerance=pd.Timedelta('5min'),
    direction='nearest'
).set_index('date')

print(f'\nCombined shape: {combined.shape}')
print(f'Combined coverage: {100 * combined.notna().sum().sum() / combined.size:.1f}%')

combined.head()